# 실전 12-3: Loop Engineering (자율 제어 반복문)

## 1. Loop Engineering이란?
- 기존에는 사용자가 프롬프트를 치면 챗봇이 한 번 답하고 끝났습니다 (단방향).
- **Loop Engineering**은 에이전트가 결과를 도출한 뒤, 그것을 '검증기(Verifier)'나 '테스트 코드'에 던져보고 에러가 나면 **성공할 때까지 스스로 코드를 수정하며 무한 반복(Self-Correction Loop)하는 자율 시스템**을 설계하는 기술입니다.
- 2026년 기준, 인간의 개입(Human-in-the-loop)을 완전히 배제하고 AI가 혼자 밤새 일해서 결과물을 내놓게 만드는 궁극의 엔지니어링 패러다임입니다.

In [1]:
import os
from typing import Annotated, TypedDict
import operator
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 2. 상태(State) 및 노드(Node) 설계
AI가 코드를 짜는 노드(Coder)와 코드를 검증하는 노드(Verifier)를 만듭니다.

In [2]:
class LoopState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]
    code: str
    error: str
    attempts: int # 무한 루프 방지용 카운터

# 1. Coder (코딩하는 AI)
def coder_node(state: LoopState):
    # 만약 이전에 에러가 났었다면, 에러 로그를 프롬프트에 추가해서 고치게 만듭니다.
    system_prompt = "당신은 천재 파이썬 개발자입니다. 주어진 문제에 대한 완벽한 파이썬 함수를 작성하세요. 오직 파이썬 코드만 출력하세요(마크다운 블록 제외)."
    if state.get("error"):
        system_prompt += f"\n\n주의! 이전 코드 실행 중 다음 에러가 발생했습니다. 원인을 분석하고 고치세요:\n{state['error']}"

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm.invoke(messages)
    
    # 코드 텍스트만 추출 (```python 과 ``` 제거)
    clean_code = response.content.replace("```python", "").replace("```", "").strip()
    return {"code": clean_code, "attempts": state.get("attempts", 0) + 1}

# 2. Verifier (실행 및 검증 환경)
def verifier_node(state: LoopState):
    code_to_test = state["code"]
    
    # 파이썬의 exec 함수를 사용해 로컬 메모리 상에서 AI가 짠 코드를 강제로 실행해봅니다.
    # 실무에서는 이 부분을 Docker 컨테이너나 Sandbox 환경으로 안전하게 분리(Harness)합니다.
    local_env = {}
    try:
        exec(code_to_test, {}, local_env)
        
        # 우리가 요구한 함수 이름이 'calculate_sum' 이라고 가정하고 테스트 실행
        if 'calculate_sum' in local_env:
            result = local_env['calculate_sum']([1, 2, 3, 4, 5])
            if result == 15:
                return {"error": ""} # 에러 없음! 통과!
            else:
                return {"error": f"로직 에러: 결과가 15여야 하는데 {result}가 나옴"}
        else:
            return {"error": "'calculate_sum' 이라는 이름의 함수를 찾을 수 없음"}
            
    except Exception as e:
        return {"error": f"문법/실행 에러: {str(e)}"}

## 3. 라우팅 로직 (무한 루프 방지 포함)
검증기가 뱉은 에러(`state['error']`)가 있는지 없는지에 따라 통과/재시도를 결정합니다.

In [3]:
def check_result(state: LoopState):
    if state["attempts"] > 5:
        print("⚠️ [Supervisor]: 시도 횟수 초과(5회). 루프를 강제 종료합니다.")
        return END
        
    if state["error"] == "":
        print("✅ [Supervisor]: 검증 완벽 통과! 최종 코드를 배포합니다.")
        return END
    else:
        print(f"❌ [Supervisor]: 에러 발생 ({state['error']}). Coder에게 다시 돌려보냅니다.")
        return "Coder"

## 4. LangGraph 조립 및 자율 반복(Self-Correction) 실행

In [4]:
workflow = StateGraph(LoopState)

workflow.add_node("Coder", coder_node)
workflow.add_node("Verifier", verifier_node)

workflow.set_entry_point("Coder")
workflow.add_edge("Coder", "Verifier")
workflow.add_conditional_edges("Verifier", check_result)

graph = workflow.compile()

print("=== [루프 엔지니어링 실행] ===")
# 일부러 AI가 틀리기 쉬운 헷갈리는 프롬프트를 줍니다.
inputs = {
    "messages": [HumanMessage(content="숫자 리스트를 받아서 합을 구하는 calculate_sum 함수를 짜줘. 단, 숫자가 아니라 문자열로 숫자가 들어와도 알아서 int로 변환해서 합산해야 해.")],
    "attempts": 0
}

for event in graph.stream(inputs):
    for node_name, state in event.items():
        if node_name == "Coder":
            print(f"\n[Coder가 짠 코드 (시도 {state['attempts']}회)]\n{state['code']}")
        elif node_name == "Verifier":
            pass # 라우팅 함수에서 상태를 프린트하므로 생략

=== [루프 엔지니어링 실행] ===

[Coder가 짠 코드 (시도 1회)]
def calculate_sum(numbers):
    total = 0
    for num in numbers:
        total += int(num)
    return total
✅ [Supervisor]: 검증 완벽 통과! 최종 코드를 배포합니다.


## 5. 결과 해석 및 의의
- AI가 처음에 짠 코드가 Verifier(검증기)에서 에러가 나면, 사람이 "너 이거 틀렸어 고쳐와" 라고 칠 필요가 없습니다.
- LangGraph의 상태(State) 시스템이 에러 로그를 그대로 들고 다시 Coder 노드로 넘어가며, Coder는 그 에러 로그를 반영하여 **스스로 코드를 고쳐서 다시 제출**합니다.
- 이것이 최근 모든 AI 기업이 추구하는 **자율 검증 및 자가 수정 루프(Self-Correction Loop)**이며, 가장 고차원적인 수준의 Loop Engineering 입니다.